In [1]:
!pip install -q pyclipper shapely editdistance


In [2]:
# ============================================================
# Imports, paths, reproducibility (SEED=42)
# ============================================================
import os, glob, re, json, random, unicodedata
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.models import resnet18, ResNet18_Weights
from tqdm import tqdm

import pyclipper
from shapely.geometry import Polygon

BASE      = "/kaggle/input/datasets/hungkhoi/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext"
TRAIN_DIR = os.path.join(BASE, "train_images")
VAL_DIR   = os.path.join(BASE, "val_image")
TEST_DIR  = os.path.join(BASE, "test_image")
LABEL_DIR = os.path.join(BASE, "labels")
OUT_DIR   = "/kaggle/working/dbnet_det_vintext"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  SEED: {SEED}")


Device: cuda  |  SEED: 42


In [3]:
# ============================================================
# Vietnamese alphabet (NFD, 106 chars + CTC blank)
# Same as crnn-dbnet-end-to-end(12epochs).ipynb — paper Sec. 5.3
# ============================================================
SPECIAL_BLANK = "<BLANK>"

VN_BASE_CHARS = (
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    " !\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"
    "\u0301"   # acute
    "\u0300"   # grave
    "\u0309"   # hook above
    "\u0323"   # dot below
    "\u0303"   # tilde
    "\u0302"   # circumflex
    "\u0306"   # breve
    "\u031b"   # horn
    "\u0111"   # d with stroke
    "\u0110"   # D with stroke
)

_seen, _char_set = set(), []
for c in VN_BASE_CHARS:
    if c not in _seen:
        _char_set.append(c); _seen.add(c)

CHARS       = [SPECIAL_BLANK] + _char_set
CHAR2IDX    = {c: i for i, c in enumerate(CHARS)}
IDX2CHAR    = {i: c for c, i in CHAR2IDX.items()}
NUM_CLASSES = len(CHARS)
print(f"Alphabet size (incl. CTC blank): {NUM_CLASSES}")


def text_to_indices(text: str) -> List[int]:
    text = unicodedata.normalize("NFD", text.strip())
    return [CHAR2IDX[c] for c in text if c in CHAR2IDX]


def indices_to_text(indices) -> str:
    return "".join(IDX2CHAR.get(i, "") for i in indices if i != 0)


def normalize_for_match(s: str) -> str:
    return unicodedata.normalize("NFD", s).strip()


Alphabet size (incl. CTC blank): 106


In [4]:
# ============================================================
# VinText annotation parser
# ============================================================
def parse_annotation(gt_path: str) -> List[Tuple[np.ndarray, str]]:
    records = []
    if not os.path.exists(gt_path):
        return records
    with open(gt_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",", 8)
            if len(parts) < 9:
                continue
            try:
                coords = list(map(float, parts[:8]))
            except ValueError:
                continue
            transcript = parts[8].strip()
            pts = np.array(coords, dtype=np.float32).reshape(4, 2)
            records.append((pts, transcript))
    return records


def _img_idx_from_name(name: str):
    m = re.search(r"(\d+)", os.path.basename(name))
    return int(m.group(1)) if m else None


def gather_image_records(img_dir: str, label_dir: str):
    for ip in sorted(glob.glob(os.path.join(img_dir, "*.jpg")) +
                     glob.glob(os.path.join(img_dir, "*.png"))):
        idx = _img_idx_from_name(ip)
        gt = os.path.join(label_dir, f"gt_{idx}.txt")
        records = parse_annotation(gt)
        if records:
            yield ip, records


In [5]:
# ============================================================
# 4. DBNet ground-truth generation
#    Implements the prob/threshold/mask map construction from
#    Liao et al., "Real-time Scene Text Detection with
#    Differentiable Binarization" (AAAI'20).
# ============================================================
def shrink_polygon(poly: np.ndarray, shrink_ratio: float = 0.4) -> Optional[np.ndarray]:
    """Vatti-clip shrink, used to build the prob-map GT (text core)."""
    try:
        p = Polygon(poly)
    except Exception:
        return None
    if (not p.is_valid) or p.area <= 1.0:
        return None
    distance = p.area * (1.0 - shrink_ratio ** 2) / max(p.length, 1e-6)
    pco = pyclipper.PyclipperOffset()
    try:
        pco.AddPath(np.round(poly).astype(int).tolist(),
                    pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
        shrunk = pco.Execute(-distance)
    except Exception:
        return None
    if not shrunk:
        return None
    return np.asarray(shrunk[0], dtype=np.int32)


def make_db_targets(polygons: List[np.ndarray],
                    transcripts: List[str],
                    h: int, w: int,
                    shrink_ratio: float = 0.4,
                    thresh_min: float = 0.3,
                    thresh_max: float = 0.7):
    """Build DBNet ground-truth maps for one image.

    Returns
    -------
    gt_prob       : (H, W) float32  -- 1 inside shrunk polygons, 0 elsewhere
    gt_thresh     : (H, W) float32  -- distance-decayed threshold map in [thresh_min, thresh_max]
    prob_mask     : (H, W) float32  -- 0 = ignore (e.g. ### regions), 1 = trainable
    thresh_mask   : (H, W) float32  -- 0/1 mask for the L1 threshold loss (only the boundary band)
    """
    gt_prob     = np.zeros((h, w), dtype=np.float32)
    gt_thresh   = np.zeros((h, w), dtype=np.float32)
    prob_mask   = np.ones ((h, w), dtype=np.float32)
    thresh_mask = np.zeros((h, w), dtype=np.float32)

    for poly, txt in zip(polygons, transcripts):
        if poly is None:
            continue
        ignore = txt in ("###", "***", "")
        if ignore:
            cv2.fillPoly(prob_mask, [poly.astype(np.int32)], 0.0)
            continue

        # ---- prob map: shrunk polygon core ----
        shrunk = shrink_polygon(poly, shrink_ratio)
        if shrunk is None:
            continue
        cv2.fillPoly(gt_prob, [shrunk.astype(np.int32)], 1.0)

        # ---- threshold map: distance band between shrunk and dilated polygon ----
        try:
            p = Polygon(poly)
            distance = p.area * (1.0 - shrink_ratio ** 2) / max(p.length, 1e-6)
        except Exception:
            continue
        pco = pyclipper.PyclipperOffset()
        try:
            pco.AddPath(np.round(poly).astype(int).tolist(),
                        pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
            dilated = pco.Execute(distance)
        except Exception:
            continue
        if not dilated:
            continue
        dilated_arr = np.asarray(dilated[0], dtype=np.int32)
        x_min, y_min = dilated_arr.min(axis=0)
        x_max, y_max = dilated_arr.max(axis=0)
        x_min = max(int(x_min), 0); y_min = max(int(y_min), 0)
        x_max = min(int(x_max), w - 1); y_max = min(int(y_max), h - 1)
        if x_max <= x_min or y_max <= y_min:
            continue
        xs = np.arange(x_min, x_max + 1)
        ys = np.arange(y_min, y_max + 1)
        xv, yv = np.meshgrid(xs, ys)

        # Distance from each pixel in the dilated bbox to the nearest polygon edge.
        n = poly.shape[0]
        dist_map = np.full(xv.shape, np.inf, dtype=np.float32)
        for k in range(n):
            x1, y1 = poly[k]
            x2, y2 = poly[(k + 1) % n]
            dx, dy = x2 - x1, y2 - y1
            seg_len_sq = dx * dx + dy * dy + 1e-12
            t = ((xv - x1) * dx + (yv - y1) * dy) / seg_len_sq
            t = np.clip(t, 0.0, 1.0)
            proj_x = x1 + t * dx
            proj_y = y1 + t * dy
            d = np.sqrt((xv - proj_x) ** 2 + (yv - proj_y) ** 2)
            dist_map = np.minimum(dist_map, d)
        dist_map = 1.0 - np.clip(dist_map / max(distance, 1e-6), 0.0, 1.0)

        # Restrict the threshold-loss band to the dilated polygon only.
        band_mask = np.zeros_like(dist_map, dtype=np.float32)
        cv2.fillPoly(band_mask, [dilated_arr - np.array([[x_min, y_min]])], 1.0)
        dist_map = dist_map * band_mask

        np.maximum(gt_thresh[y_min:y_max + 1, x_min:x_max + 1],
                   dist_map,
                   out=gt_thresh[y_min:y_max + 1, x_min:x_max + 1])
        thresh_mask[y_min:y_max + 1, x_min:x_max + 1] = np.maximum(
            thresh_mask[y_min:y_max + 1, x_min:x_max + 1], band_mask
        )

    gt_thresh = gt_thresh * (thresh_max - thresh_min) + thresh_min
    return gt_prob, gt_thresh, prob_mask, thresh_mask


In [6]:
# ============================================================
# 5. Image preprocessing utilities + helpers shared by both stages
# ============================================================
DET_INPUT_SIZE = 640                              # DBNet input resolution
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def letterbox_image(img: np.ndarray, size: int = DET_INPUT_SIZE):
    """Resize keeping aspect ratio, then pad to (size, size).

    Returns: (padded_image, scale, pad_x, pad_y)
    """
    h, w = img.shape[:2]
    scale = size / max(h, w)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((size, size, 3), dtype=resized.dtype)
    pad_x = (size - new_w) // 2
    pad_y = (size - new_h) // 2
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
    return canvas, scale, pad_x, pad_y


def transform_polygons(polys: List[np.ndarray], scale: float, pad_x: int, pad_y: int) -> List[np.ndarray]:
    out = []
    for p in polys:
        q = p.copy().astype(np.float32)
        q[:, 0] = q[:, 0] * scale + pad_x
        q[:, 1] = q[:, 1] * scale + pad_y
        out.append(q)
    return out


def to_tensor_norm(img_bgr: np.ndarray) -> torch.Tensor:
    """BGR uint8 -> ImageNet-normalised float CHW tensor (used by DBNet)."""
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - MEAN) / STD
    return torch.from_numpy(img.transpose(2, 0, 1)).contiguous()


def order_quad(pts: np.ndarray) -> np.ndarray:
    """Reorder 4 points as [TL, TR, BR, BL] so they line up with crop_quad's expectations."""
    pts = np.asarray(pts, dtype=np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]              # TL
    rect[2] = pts[np.argmax(s)]              # BR
    diff = (pts[:, 1] - pts[:, 0])
    rect[1] = pts[np.argmin(diff)]           # TR
    rect[3] = pts[np.argmax(diff)]           # BL
    return rect


# Recognition crop sizes (kept identical to the recognition-only notebook)
IMG_H = 32
IMG_W_MAX = 512


def crop_quad(image_bgr: np.ndarray, pts: np.ndarray, target_h: int = IMG_H) -> np.ndarray:
    """Perspective-warp a 4-point quadrilateral to a target_h x W strip."""
    pts = pts.astype(np.float32)
    w1 = np.linalg.norm(pts[1] - pts[0]); w2 = np.linalg.norm(pts[2] - pts[3])
    h1 = np.linalg.norm(pts[3] - pts[0]); h2 = np.linalg.norm(pts[2] - pts[1])
    W = max(int((w1 + w2) / 2), 1); H = max(int((h1 + h2) / 2), 1)
    dst = np.array([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(pts, dst)
    crop = cv2.warpPerspective(image_bgr, M, (W, H))
    scale = target_h / max(crop.shape[0], 1)
    new_w = max(int(crop.shape[1] * scale), 1)
    return cv2.resize(crop, (new_w, target_h))


def preprocess_crop(crop_bgr: np.ndarray) -> torch.Tensor:
    """RGB -> [-1, 1] CHW tensor (matches the recognition-only CRNN notebook)."""
    img = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    return torch.from_numpy(img.transpose(2, 0, 1))


In [7]:
# ============================================================
# Detection dataset (full images)
# ============================================================
class SceneTextDataset(Dataset):
    def __init__(self, img_dir: str, label_dir: str, det_size: int = DET_INPUT_SIZE):
        self.det_size = det_size
        self.records  = []
        for ip, recs in gather_image_records(img_dir, label_dir):
            polys, txts = zip(*recs)
            self.records.append((ip, list(polys), list(txts)))
        print(f"  [det] {os.path.basename(img_dir)}: {len(self.records)} images")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        ip, polys, txts = self.records[idx]
        img = cv2.imread(ip)
        if img is None:
            img = np.zeros((self.det_size, self.det_size, 3), dtype=np.uint8)
            polys, txts = [], []
        img, scale, px, py = letterbox_image(img, self.det_size)
        polys_resized = transform_polygons(polys, scale, px, py)
        gt_prob, gt_thresh, prob_mask, thresh_mask = make_db_targets(
            polys_resized, txts, self.det_size, self.det_size,
        )
        return (
            to_tensor_norm(img),
            torch.from_numpy(gt_prob),
            torch.from_numpy(gt_thresh),
            torch.from_numpy(prob_mask),
            torch.from_numpy(thresh_mask),
        )


In [8]:
# ============================================================
# 7. DBNet model: ResNet-18 + FPN + dual heads (prob & threshold)
#    Differentiable binarization (Liao et al., AAAI'20).
# ============================================================
class FPN(nn.Module):
    """Lightweight FPN matched to ResNet-18 stage channels (64,128,256,512)."""
    def __init__(self, in_channels=(64, 128, 256, 512), inner_channels: int = 256):
        super().__init__()
        c2, c3, c4, c5 = in_channels
        self.lat5 = nn.Conv2d(c5, inner_channels, 1)
        self.lat4 = nn.Conv2d(c4, inner_channels, 1)
        self.lat3 = nn.Conv2d(c3, inner_channels, 1)
        self.lat2 = nn.Conv2d(c2, inner_channels, 1)
        self.smooth5 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth4 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth3 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)
        self.smooth2 = nn.Conv2d(inner_channels, inner_channels // 4, 3, padding=1)

    def forward(self, c2, c3, c4, c5):
        p5 = self.lat5(c5)
        p4 = self.lat4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode="bilinear", align_corners=False)
        p3 = self.lat3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="bilinear", align_corners=False)
        p2 = self.lat2(c2) + F.interpolate(p3, size=c2.shape[-2:], mode="bilinear", align_corners=False)
        p5 = self.smooth5(p5); p4 = self.smooth4(p4)
        p3 = self.smooth3(p3); p2 = self.smooth2(p2)
        size = p2.shape[-2:]
        p3 = F.interpolate(p3, size=size, mode="bilinear", align_corners=False)
        p4 = F.interpolate(p4, size=size, mode="bilinear", align_corners=False)
        p5 = F.interpolate(p5, size=size, mode="bilinear", align_corners=False)
        return torch.cat([p2, p3, p4, p5], dim=1)        # (B, inner, H/4, W/4)


class DBNet(nn.Module):
    def __init__(self, k: int = 50, pretrained: bool = True):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        backbone = resnet18(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1   # /4
        self.layer2 = backbone.layer2   # /8
        self.layer3 = backbone.layer3   # /16
        self.layer4 = backbone.layer4   # /32
        self.fpn = FPN()
        inner = 256

        def _head():
            return nn.Sequential(
                nn.Conv2d(inner, inner // 4, 3, padding=1, bias=False),
                nn.BatchNorm2d(inner // 4), nn.ReLU(True),
                nn.ConvTranspose2d(inner // 4, inner // 4, 2, stride=2),
                nn.BatchNorm2d(inner // 4), nn.ReLU(True),
                nn.ConvTranspose2d(inner // 4, 1, 2, stride=2),
            )

        self.prob_head   = _head()
        self.thresh_head = _head()
        self.k = k

    def forward(self, x):
        c1 = self.stem(x)
        c2 = self.layer1(c1)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        feat   = self.fpn(c2, c3, c4, c5)
        prob   = torch.sigmoid(self.prob_head(feat))         # (B, 1, H, W)
        thresh = torch.sigmoid(self.thresh_head(feat))       # (B, 1, H, W)
        binary = torch.sigmoid(self.k * (prob - thresh))     # differentiable binarization
        return prob, thresh, binary


In [9]:
# ============================================================
# 8. DBNet loss = BCE+OHEM (prob)  +  L1 (threshold)  +  Dice (binary)
# ============================================================
class BalanceBCELoss(nn.Module):
    """BCE with online hard negative mining (DBNet uses 3:1 neg:pos ratio)."""
    def __init__(self, neg_ratio: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.neg_ratio = neg_ratio
        self.eps = eps

    def forward(self, pred, gt, mask):
        positive = (gt * mask).bool()
        negative = ((1 - gt) * mask).bool()
        n_pos = int(positive.float().sum().item())
        n_neg = min(int(negative.float().sum().item()), int(n_pos * self.neg_ratio))
        if n_pos == 0:
            return pred.sum() * 0.0
        loss = F.binary_cross_entropy(pred, gt, reduction="none")
        pos_loss = loss[positive].sum()
        if n_neg > 0:
            neg_loss = loss[negative]
            neg_loss, _ = neg_loss.topk(n_neg)
            neg_loss = neg_loss.sum()
        else:
            neg_loss = pred.sum() * 0.0
        return (pos_loss + neg_loss) / (n_pos + n_neg + self.eps)


class DiceLoss(nn.Module):
    def __init__(self, eps: float = 1e-6):
        super().__init__(); self.eps = eps

    def forward(self, pred, gt, mask):
        pred = pred * mask
        gt   = gt   * mask
        inter = (pred * gt).sum()
        return 1.0 - (2 * inter + self.eps) / (pred.sum() + gt.sum() + self.eps)


class L1MaskLoss(nn.Module):
    def forward(self, pred, gt, mask):
        if mask.sum() == 0:
            return pred.sum() * 0.0
        return (torch.abs(pred - gt) * mask).sum() / mask.sum()


class DBLoss(nn.Module):
    """Total loss for DBNet: alpha * BCE + beta * L1 + Dice."""
    def __init__(self, alpha: float = 1.0, beta: float = 10.0):
        super().__init__()
        self.bce  = BalanceBCELoss()
        self.dice = DiceLoss()
        self.l1   = L1MaskLoss()
        self.alpha = alpha
        self.beta  = beta

    def forward(self, prob, thresh, binary,
                gt_prob, gt_thresh, prob_mask, thresh_mask):
        prob   = prob.squeeze(1)
        thresh = thresh.squeeze(1)
        binary = binary.squeeze(1)
        loss_p = self.bce(prob, gt_prob, prob_mask)
        loss_t = self.l1(thresh, gt_thresh, thresh_mask)
        loss_b = self.dice(binary, gt_prob, prob_mask)
        total  = self.alpha * loss_p + self.beta * loss_t + loss_b
        return total, dict(prob=loss_p.item(), thresh=loss_t.item(), bin=loss_b.item())


In [10]:
# ============================================================
# 9. DBNet post-processing: prob map -> 4-point quad polygons
# ============================================================
@torch.no_grad()
def decode_db_polygons(prob_map: torch.Tensor,
                       prob_thresh: float = 0.3,
                       box_thresh: float = 0.55,
                       min_size: float = 3.0,
                       max_candidates: int = 1000,
                       unclip_ratio: float = 1.5):
    """Convert DBNet probability maps to lists of (quad, score) per image.

    Coordinates are returned in the detector's input space (det_size x det_size).
    """
    results = []
    prob_np = prob_map.detach().squeeze(1).cpu().numpy()      # (B, H, W)
    binary  = (prob_np >= prob_thresh).astype(np.uint8)
    for prob, b in zip(prob_np, binary):
        contours, _ = cv2.findContours(b, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        polys = []
        for cnt in contours[:max_candidates]:
            if cnt.shape[0] < 4:
                continue
            epsilon = 0.005 * cv2.arcLength(cnt, True)
            approx  = cv2.approxPolyDP(cnt, epsilon, True).reshape(-1, 2)
            if approx.shape[0] < 4:
                rect = cv2.minAreaRect(cnt)
                approx = cv2.boxPoints(rect).astype(np.float32)
            mask = np.zeros_like(prob, dtype=np.uint8)
            cv2.fillPoly(mask, [approx.astype(np.int32)], 1)
            if mask.sum() == 0:
                continue
            score = float(prob[mask == 1].mean())
            if score < box_thresh:
                continue
            try:
                p = Polygon(approx.astype(np.float64))
                if (not p.is_valid) or p.area < min_size:
                    continue
                distance = p.area * unclip_ratio / max(p.length, 1e-6)
            except Exception:
                continue
            pco = pyclipper.PyclipperOffset()
            try:
                pco.AddPath(np.round(approx).astype(int).tolist(),
                            pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
                expanded = pco.Execute(distance)
            except Exception:
                continue
            if not expanded:
                continue
            poly = np.asarray(expanded[0], dtype=np.float32)
            if poly.shape[0] < 4:
                continue
            # Reduce to a 4-point quadrilateral so it lines up with crop_quad's expectation.
            rect = cv2.minAreaRect(poly)
            quad = cv2.boxPoints(rect).astype(np.float32)
            quad = order_quad(quad)
            polys.append((quad, score))
        results.append(polys)
    return results


In [11]:
# ============================================================
# Detection-only evaluation (IoU >= 0.5, greedy matching)
# ============================================================
def polygon_iou(p1: np.ndarray, p2: np.ndarray) -> float:
    try:
        a = Polygon(p1); b = Polygon(p2)
        if not a.is_valid: a = a.buffer(0)
        if not b.is_valid: b = b.buffer(0)
        if a.is_empty or b.is_empty:
            return 0.0
        inter = a.intersection(b).area
        union = a.union(b).area
        return float(inter / union) if union > 0 else 0.0
    except Exception:
        return 0.0


def _prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    h = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, h


@torch.no_grad()
def evaluate_detection(detector: DBNet,
                       img_dir: str, label_dir: str,
                       iou_thresh: float = 0.5,
                       max_images: Optional[int] = None,
                       prob_thresh: float = 0.3,
                       box_thresh: float = 0.55,
                       unclip_ratio: float = 1.5) -> dict:
    detector.eval()
    image_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")) +
                         glob.glob(os.path.join(img_dir, "*.png")))
    if max_images:
        image_paths = image_paths[:max_images]

    det_TP = det_FP = det_FN = 0
    n_images = 0

    for ip in tqdm(image_paths, desc=f"Det eval [{os.path.basename(img_dir)}]"):
        idx = _img_idx_from_name(ip)
        gt = parse_annotation(os.path.join(label_dir, f"gt_{idx}.txt"))
        if not gt:
            continue
        gt_polys = [g[0] for g in gt]
        gt_texts = [normalize_for_match(g[1]) for g in gt]
        gt_dontcare = [t in ("###", "***", "") for t in gt_texts]

        img = cv2.imread(ip)
        if img is None:
            det_FN += sum(1 for d in gt_dontcare if not d)
            continue

        h0, w0 = img.shape[:2]
        canvas, scale, px, py = letterbox_image(img, DET_INPUT_SIZE)
        tensor = to_tensor_norm(canvas).unsqueeze(0).to(DEVICE)
        prob, _, _ = detector(tensor)
        decoded = decode_db_polygons(
            prob, prob_thresh, box_thresh, unclip_ratio=unclip_ratio,
        )[0]

        preds = []
        for quad, _score in decoded:
            q = quad.copy()
            q[:, 0] = (q[:, 0] - px) / max(scale, 1e-6)
            q[:, 1] = (q[:, 1] - py) / max(scale, 1e-6)
            q[:, 0] = np.clip(q[:, 0], 0, w0 - 1)
            q[:, 1] = np.clip(q[:, 1], 0, h0 - 1)
            preds.append(q.astype(np.float32))

        n_images += 1
        gt_used = [False] * len(gt_polys)
        for poly in preds:
            best_iou = 0.0; best_j = -1
            for j, gp in enumerate(gt_polys):
                if gt_used[j]:
                    continue
                iou = polygon_iou(poly, gp)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            matched = (best_j >= 0 and best_iou >= iou_thresh)
            if matched and gt_dontcare[best_j]:
                gt_used[best_j] = True
                continue
            if matched:
                gt_used[best_j] = True
                det_TP += 1
            else:
                det_FP += 1
        for j, used in enumerate(gt_used):
            if used or gt_dontcare[j]:
                continue
            det_FN += 1

    P, R, H = _prf(det_TP, det_FP, det_FN)
    return dict(
        precision=P, recall=R, hmean=H,
        det_precision=P, det_recall=R, det_hmean=H,
        det_TP=det_TP, det_FP=det_FP, det_FN=det_FN,
        images=n_images,
    )


def print_det_results(tag: str, res: dict):
    print(f"  [{tag}] Det P={res['precision']:.4f}  R={res['recall']:.4f}  H-mean={res['hmean']:.4f}  "
          f"(TP={res['det_TP']} FP={res['det_FP']} FN={res['det_FN']} over {res['images']} images)")


In [12]:
# ============================================================
# Detection-only training
# ============================================================
METRIC_SUBSET_TRAIN = 60   # images for per-epoch train det H-mean (speed)
METRIC_SUBSET_VAL   = 80   # images for per-epoch val det H-mean


def train_detection(detector: DBNet,
                    train_loader, val_loader,
                    num_epochs: int = 12,
                    lr: float = 1e-4,
                    grad_clip: float = 5.0,
                    ckpt_path: Optional[str] = None):
    db_loss_fn = DBLoss().to(DEVICE)
    optimizer = AdamW(detector.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    history = dict(train_loss=[], val_loss=[], train_acc=[], val_acc=[])
    if ckpt_path is None:
        ckpt_path = os.path.join(OUT_DIR, "dbnet_det_best.pth")

    best_metric = -1.0
    for epoch in range(1, num_epochs + 1):
        detector.train()
        running = 0.0
        pbar = tqdm(train_loader, desc=f"E{epoch}/{num_epochs} det train")
        for batch in pbar:
            imgs, gtp, gtt, pmask, tmask = [x.to(DEVICE, non_blocking=True) for x in batch]
            prob, thresh, binary = detector(imgs)
            loss, parts = db_loss_fn(prob, thresh, binary, gtp, gtt, pmask, tmask)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(detector.parameters(), grad_clip)
            optimizer.step()
            running += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.3f}", prob=f"{parts['prob']:.3f}")
        scheduler.step()
        train_loss = running / max(len(train_loader), 1)
        history["train_loss"].append(train_loss)

        detector.eval()
        v_loss = 0.0; steps = 0
        with torch.no_grad():
            for batch in val_loader:
                imgs, gtp, gtt, pmask, tmask = [x.to(DEVICE) for x in batch]
                prob, thresh, binary = detector(imgs)
                dl, _ = db_loss_fn(prob, thresh, binary, gtp, gtt, pmask, tmask)
                v_loss += dl.item(); steps += 1
        v_loss /= max(steps, 1)
        history["val_loss"].append(v_loss)

        train_h = evaluate_detection(
            detector, TRAIN_DIR, LABEL_DIR, max_images=METRIC_SUBSET_TRAIN,
        )["hmean"]
        val_h = evaluate_detection(
            detector, VAL_DIR, LABEL_DIR, max_images=METRIC_SUBSET_VAL,
        )["hmean"]
        history["train_acc"].append(train_h)
        history["val_acc"].append(val_h)

        track = val_h
        print(f"Epoch {epoch}: train_loss={train_loss:.4f}  val_loss={v_loss:.4f}  "
              f"| train_det_H={train_h:.4f}  val_det_H={val_h:.4f}")

        if track > best_metric:
            best_metric = track
            torch.save({
                "epoch": epoch,
                "seed": SEED,
                "framework": "pytorch_dbnet_resnet18_vintext",
                "det_input_size": DET_INPUT_SIZE,
                "num_classes_rec": NUM_CLASSES,
                "detector_state": detector.state_dict(),
                "history": history,
                "best_metric": best_metric,
            }, ckpt_path)
            print(f"    saved best -> {ckpt_path}")

    return history, ckpt_path


In [13]:
# ============================================================
# Build data, train, export .pth for downstream E2E notebook
# ============================================================
print("=== Building detection datasets ===")
det_train = SceneTextDataset(TRAIN_DIR, LABEL_DIR)
det_val   = SceneTextDataset(VAL_DIR,   LABEL_DIR)

det_train_loader = DataLoader(det_train, batch_size=4, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
det_val_loader   = DataLoader(det_val, batch_size=4, shuffle=False,
                              num_workers=2, pin_memory=True)

print(f"Det batches: train={len(det_train_loader)}  val={len(det_val_loader)}")

detector = DBNet(pretrained=True).to(DEVICE)
print(f"Detector params: {sum(p.numel() for p in detector.parameters()):,}")

NUM_EPOCHS = 12
CKPT_BEST = os.path.join(OUT_DIR, "dbnet_det_best.pth")

print("\n=== Training DBNet (detection only, 12 epochs) ===")
history, best_path = train_detection(
    detector, det_train_loader, det_val_loader,
    num_epochs=NUM_EPOCHS, ckpt_path=CKPT_BEST,
)

# Export portable .pth (upload to Kaggle Models for E2E notebook)
PTH_EXPORT = os.path.join(OUT_DIR, "dbnet_resnet18_det_vintext_12epochs.pth")
ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
torch.save(ckpt, PTH_EXPORT)
torch.save(ckpt["detector_state"], os.path.join(OUT_DIR, "dbnet_resnet18_det_vintext_12epochs_state_only.pth"))
print(f"\nExported detector bundle -> {PTH_EXPORT}")
print("Upload this file as a Kaggle dataset; use path in dbnet-crnn-e2e-vintext-12epochs.ipynb")


=== Building detection datasets ===
  [det] train_images: 1200 images
  [det] val_image: 300 images
Det batches: train=300  val=75
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


Detector params: 12,342,210

=== Training DBNet (detection only, 12 epochs) ===


Det eval [val_image]: 100%|██████████| 80/80 [00:15<00:00,  5.15it/s]


Epoch 1: train_loss=2.1928  val_loss=1.7774  | train_det_H=0.0000  val_det_H=0.0000
    saved best -> /kaggle/working/dbnet_det_vintext/dbnet_det_best.pth


...

Epoch 10: train_loss=0.8020  val_loss=1.1200  | train_det_H=0.6530  val_det_H=0.6488
    saved best -> /kaggle/working/dbnet_det_vintext/dbnet_det_best.pth


Det eval [val_image]: 100%|██████████| 80/80 [00:09<00:00,  8.07it/s]


Epoch 11: train_loss=0.7820  val_loss=1.1193  | train_det_H=0.6705  val_det_H=0.6517
    saved best -> /kaggle/working/dbnet_det_vintext/dbnet_det_best.pth


Det eval [val_image]: 100%|██████████| 80/80 [00:09<00:00,  8.05it/s]


Epoch 12: train_loss=0.7670  val_loss=1.1166  | train_det_H=0.6651  val_det_H=0.6569
    saved best -> /kaggle/working/dbnet_det_vintext/dbnet_det_best.pth

Exported detector bundle -> /kaggle/working/dbnet_det_vintext/dbnet_resnet18_det_vintext_12epochs.pth
Upload this file as a Kaggle dataset; use path in dbnet-crnn-e2e-vintext-12epochs.ipynb


In [14]:
# ============================================================
# Training curves — loss & detection H-mean (train/val subsets)
# ============================================================
_hist = history if "history" in dir() and history.get("train_loss") else None
if _hist is None:
    _hist = torch.load(best_path, map_location="cpu", weights_only=False).get("history", {})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(_hist["train_loss"]) + 1)

axes[0].plot(ep, _hist["train_loss"], "o-", label="train", linewidth=2, markersize=5)
axes[0].plot(ep, _hist["val_loss"], "o-", label="val", linewidth=2, markersize=5)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("DB loss")
axes[0].set_title("Train / Val loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, _hist["train_acc"], "o-", label="train", linewidth=2, markersize=5)
axes[1].plot(ep, _hist["val_acc"], "o-", label="val", linewidth=2, markersize=5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Detection H-mean")
axes[1].set_title(f"Train / Val accuracy (Det H-mean, subset)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = os.path.join(OUT_DIR, "training_curves.png")
plt.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Training curves saved -> {curve_path}")


Training curves saved -> /kaggle/working/dbnet_det_vintext/training_curves.png


In [15]:
# ============================================================
# Final detection metrics (val + test)
# ============================================================
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
detector.load_state_dict(ckpt["detector_state"])

final_results = {}
for tag, img_dir in [("Validation", VAL_DIR), ("Test", TEST_DIR)]:
    print(f"\n--- {tag} ---")
    res = evaluate_detection(detector, img_dir, LABEL_DIR, iou_thresh=0.5)
    final_results[tag] = res
    print_det_results(tag, res)
    with open(os.path.join(OUT_DIR, f"det_metrics_{tag.lower()}.json"), "w") as f:
        json.dump(res, f, indent=2)

print("\n" + "=" * 60)
print(f"  {'Split':<14} {'P':>8} {'R':>8} {'H-mean':>8}")
print("  " + "-" * 44)
for tag, r in final_results.items():
    print(f"  {tag:<14} {r['precision']:>8.4f} {r['recall']:>8.4f} {r['hmean']:>8.4f}")
print("=" * 60)



--- Validation ---


Det eval [val_image]: 100%|██████████| 300/300 [00:36<00:00,  8.26it/s]


  [Validation] Det P=0.8294  R=0.4842  H-mean=0.6115  (TP=3487 FP=717 FN=3714 over 300 images)

--- Test ---


Det eval [test_image]: 100%|██████████| 500/500 [01:01<00:00,  8.19it/s]

  [Test] Det P=0.8535  R=0.5301  H-mean=0.6540  (TP=5337 FP=916 FN=4730 over 500 images)

  Split                 P        R   H-mean
  --------------------------------------------
  Validation       0.8294   0.4842   0.6115
  Test             0.8535   0.5301   0.6540


In [16]:
# ============================================================
# Sample inference — GT (green) vs predicted boxes (orange)
# ============================================================
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
detector.load_state_dict(ckpt["detector_state"])
detector.eval()

GT_COLOR = (0, 200, 0)
PRED_COLOR = (0, 165, 255)
N_SAMPLES = 6
SAMPLE_DIR = TEST_DIR


def _draw_polys(canvas, polys, color, thickness=2):
    for poly in polys:
        pts = np.asarray(poly, dtype=np.int32).reshape(-1, 1, 2)
        if pts.shape[0] >= 3:
            cv2.polylines(canvas, [pts], True, color, thickness)
    return canvas


@torch.no_grad()
def _predict_polys(det_model, image_bgr):
    h0, w0 = image_bgr.shape[:2]
    canvas, scale, px, py = letterbox_image(image_bgr, DET_INPUT_SIZE)
    tensor = to_tensor_norm(canvas).unsqueeze(0).to(DEVICE)
    prob, _, _ = det_model(tensor)
    decoded = decode_db_polygons(prob)[0]
    out = []
    for quad, _ in decoded:
        q = quad.copy()
        q[:, 0] = (q[:, 0] - px) / max(scale, 1e-6)
        q[:, 1] = (q[:, 1] - py) / max(scale, 1e-6)
        q[:, 0] = np.clip(q[:, 0], 0, w0 - 1)
        q[:, 1] = np.clip(q[:, 1], 0, h0 - 1)
        out.append(q.astype(np.float32))
    return out


viz_dir = os.path.join(OUT_DIR, "det_inference_samples")
os.makedirs(viz_dir, exist_ok=True)

sample_paths = sorted(
    glob.glob(os.path.join(SAMPLE_DIR, "*.jpg")) +
    glob.glob(os.path.join(SAMPLE_DIR, "*.png"))
)[:N_SAMPLES]

for ip in sample_paths:
    img = cv2.imread(ip)
    if img is None:
        continue
    idx = _img_idx_from_name(ip)
    gt = parse_annotation(os.path.join(LABEL_DIR, f"gt_{idx}.txt"))
    gt_polys = [p for p, t in gt if t not in ("###", "***", "")]

    pred_polys = _predict_polys(detector, img)
    gt_canvas = _draw_polys(img.copy(), gt_polys, GT_COLOR, 3)
    pred_canvas = _draw_polys(img.copy(), pred_polys, PRED_COLOR, 2)

    fig, axes = plt.subplots(1, 3, figsize=(21, 7))
    for ax, vis, title in zip(
        axes,
        [img, gt_canvas, pred_canvas],
        ["Original", f"GT ({len(gt_polys)})", f"Pred ({len(pred_polys)})"],
    ):
        ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.axis("off")
    plt.suptitle(os.path.basename(ip), fontsize=13)
    plt.tight_layout()
    save_path = os.path.join(viz_dir, f"det_{Path(ip).stem}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  saved -> {save_path}")

print(f"\nDetection sample visualizations -> {viz_dir}")


  saved -> /kaggle/working/dbnet_det_vintext/det_inference_samples/det_im1501.png
  saved -> /kaggle/working/dbnet_det_vintext/det_inference_samples/det_im1502.png
  saved -> /kaggle/working/dbnet_det_vintext/det_inference_samples/det_im1503.png
  saved -> /kaggle/working/dbnet_det_vintext/det_inference_samples/det_im1504.png
  saved -> /kaggle/working/dbnet_det_vintext/det_inference_samples/det_im1505.png
  saved -> /kaggle/working/dbnet_det_vintext/det_inference_samples/det_im1506.png

Detection sample visualizations -> /kaggle/working/dbnet_det_vintext/det_inference_samples
